# GridNet-HD — Robustness protocol (paper Sec. 5): models under camera-pose error

ต้องมีก่อน: LiDAR cache + zone tar ของ **val 6 zones**, checkpoint LiDAR-only, และ `best.pth` ของ heads ที่จะวัด (`GridNet/fusion/<run>/best.pth`)

ทำอะไร: ต่อ val zone — (1) คำนวณ feature/logits ของ frozen backbone บน subset แบบ stratified ตามคลาส **ครั้งเดียว** (เก็บไว้ที่
`GridNet/protocol/subsets_<RUN2D>/<zone>/subset.npz` ใช้ซ้ำได้ทุก run) → (2) ทุก (sweep, level, draw): perturb กล้องทุกตัว → re-project/re-associate →
head forward ของทุกโมเดล (LiDAR-only, ImageVote, heads) → weighted confusion → `results.json` → (3) `summary.json` / `curves.csv` / `scores.csv` + รูป

| MODE | association ภายใต้ noise | เวลาต่อ zone-draw (A100) | ใช้เมื่อ |
|---|---|---|---|
| `fast` | re-project 8 views เดิม คง visibility ของ association เดิมไว้ (ตัดเฉพาะคู่ที่หลุดขอบภาพ) | ~1 วินาที (6 โมเดล) | ระหว่างพัฒนา head (run `fast_v1`: 6 zones × 133 evaluations ใน 18 นาที); `colab_heads_lab` เรียกโหมดนี้ให้เองท้าย lab |
| `fast_retest` | เหมือน `fast` แต่เช็ค visibility ของ pixel ใหม่กับ depth map เดิมด้วย (แบบแรกที่ออกแบบไว้) | ~1 วินาที | ใช้เทียบใน cell 8 เท่านั้น |
| `exact` | render depth map ใหม่ด้วยกล้องที่ถูก perturb + เลือก views ใหม่ | ~6 วินาที | **ตัวเลขใน paper** (ทั้ง 6 zones ประมาณ 1–1.3 ชม. ยังไม่เคยรันเต็ม) |

วัดแล้ว (run `fast_v1`, 13 พ.ย.): `fast` ต่างจาก `exact` ไม่เกิน 0.13 mIoU สำหรับ fusion heads แต่ดีเกินจริงได้ถึง 0.8 mIoU สำหรับ ImageVote
(structural 4.8 IoU ที่ 10 cm) → ตัวเลขสุดท้ายใช้ `exact`; `fast_retest` ต่างถึง 1.4 mIoU

sweep `trans` / `rot`: `DRAWS` draws ต่อ level · sweep `drift`: 2 evaluations ต่อ level (ทิศ + และ −; drift ไม่มีความสุ่มอย่างอื่น)

**Resume**: Run all ซ้ำได้ — zone ที่อยู่ใน results.json แล้วถูกข้าม แต่ต้องเป็นค่าเดิม (heads, sweeps, draws, mode) ถ้าเปลี่ยนให้ตั้ง `TAG` ใหม่ (script จะหยุดเองถ้าไม่ตรง)

In [ ]:
#@title 1. GPU (A100 asked for; an L4 also works for this notebook - only the validation zones are loaded - but is slower)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need L4 / A100 (flash-attn)"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
TAG = 'exact_v1'                                 #@param {type:"string"}
MODE = 'exact'                                   #@param ["fast", "fast_retest", "exact"]
DRAWS = 10                                       #@param {type:"integer"}
SWEEPS = 'trans,rot,drift'                       #@param {type:"string"}
HEADS = 'late_v1,concat_v1,window3_v1,late_nat_b1'   #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
BACKBONE_CKPT = f'{DRIVE}/ckpt/litept_s_lidar_v2/best.pth'   #@param {type:"string"}
VAL_ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'     #@param {type:"string"}
CAP = 500000              #@param {type:"integer"}
CAP_FULL = 2000000        #@param {type:"integer"}
EVAL_MAX_POINTS = 900000  #@param {type:"integer"}
SEED = 0                  #@param {type:"integer"}
EXACT_CHECK = True        #@param {type:"boolean"}
EXACT_ZONES = 't5b5,t1z8' #@param {type:"string"}
EXACT_SWEEPS = 'trans,rot,drift' #@param {type:"string"}
EXACT_DRAWS = 2           #@param {type:"integer"}
SHUTDOWN_WHEN_DONE = False #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
ASSOC_DIR = f'{DRIVE}/fusion/assoc_{RUN2D}'
IMAGEVOTE_DIR = f'{DRIVE}/fusion/imagevote_{RUN2D}'
SUBSET_DIR = f'{DRIVE}/protocol/subsets_{RUN2D}'
OUT = f'{DRIVE}/protocol/{TAG}'
OUT_EXACT = f'{DRIVE}/protocol/{TAG}_exact'
OUT_RETEST = f'{DRIVE}/protocol/{TAG}_retest'
for d in (ASSOC_DIR, SUBSET_DIR, OUT): os.makedirs(d, exist_ok=True)
val_zones = [z.strip() for z in VAL_ZONES.split(',') if z.strip()]
exact_zones = [z.strip() for z in EXACT_ZONES.split(',') if z.strip()]
assert MODE == 'exact' or not EXACT_CHECK or set(exact_zones) <= set(val_zones), "EXACT_ZONES must be among VAL_ZONES"
heads = {}
for name in [h.strip() for h in HEADS.split(',') if h.strip()]:
    path = f'{DRIVE}/fusion/{name}/best.pth'
    assert os.path.exists(path), f"no checkpoint for head {name}: {path}"
    heads[name] = path
print("heads:", list(heads), "\nout ->", OUT)


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow', 'matplotlib'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert 'PROTOCOL_VERSION' in open('/content/code/gridnet/train/protocol.py').read(), "gridnet_code.zip on Drive is older than update32: pack and upload it again"
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.protocol; print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Data -> local disk (val zones only): LiDAR caches, zone tars, camera files, backbone checkpoint, associations
import os, sys, shutil, subprocess, time
sys.path.insert(0, '/content/code')
from gridnet.train.maps2d import cached_image_ids, zone_tar_manifest, extract_zone_tar, write_compact
from concurrent.futures import ThreadPoolExecutor
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
zones = val_zones
t0 = time.time()
for z in zones:
    dst = f'/content/cache/{z}/points.npz'
    if not os.path.exists(dst):
        os.makedirs(f'/content/cache/{z}', exist_ok=True); shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', dst)
print(f"LiDAR caches ready ({(time.time() - t0) / 60:.1f} min)")
CACHE2D = '/content/cache2d'; t0 = time.time()
for z in zones:
    if zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None:
        n = extract_zone_tar(CACHE2D_DRIVE, z, CACHE2D); how = 'tar'
    else:
        ids = cached_image_ids(CACHE2D_DRIVE, z)
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(lambda i: write_compact(CACHE2D_DRIVE, z, i, CACHE2D), ids))
        n = len(cached_image_ids(CACHE2D, z)); how = 'files (slow: no tar yet)'
    print(f"  {z}: {n} images from {how}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True); print("camera files ready")
CKPT_LOCAL = '/content/backbone_best.pth'
if not os.path.exists(CKPT_LOCAL): shutil.copy(BACKBONE_CKPT, CKPT_LOCAL)
for z in zones:                                        # clean val associations saved by colab_fusion / colab_train_fusion
    src, dst = f'{IMAGEVOTE_DIR}/{z}/assoc.npz', f'{ASSOC_DIR}/{z}/assoc.npz'
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(f'{ASSOC_DIR}/{z}', exist_ok=True); shutil.copy(src, dst); print(f"  assoc {z}: copied from imagevote")
print("associations present:", [z for z in zones if os.path.exists(f'{ASSOC_DIR}/{z}/assoc.npz')], "(missing ones are computed on first use)")
print("subsets present:", [z for z in zones if os.path.exists(f'{SUBSET_DIR}/{z}/subset.npz')], "(missing ones are computed on first use, ~2 min each)")


In [ ]:
#@title 6. Run the protocol (resumes per zone; results -> OUT/results.json)
import subprocess, time, os
cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
       '--assoc-dir', ASSOC_DIR, '--out', OUT, '--subset-dir', SUBSET_DIR, '--litept', '/content/LitePT',
       '--backbone-ckpt', CKPT_LOCAL, '--zones', ','.join(val_zones), '--sweeps', SWEEPS, '--draws', str(DRAWS),
       '--mode', MODE, '--cap', str(CAP), '--cap-full', str(CAP_FULL), '--eval-max-points', str(EVAL_MAX_POINTS), '--seed', str(SEED),
       '--heads'] + [f'{n}={p}' for n, p in heads.items()]
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'Warning' not in line and 'def backward' not in line and '_TORCH_CUSTOM_FWD' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
    if p.returncode != 0: print("-> if it STOPPED on 'other settings': this TAG already holds a run with other heads / sweeps / draws / mode - give TAG a new name in cell 2")
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped by user; finished zones are in results.json -> re-run this cell to continue")


In [ ]:
#@title 7. Summary table + figure
import subprocess, json, os, sys
from IPython.display import Image, display
fig = f'{OUT}/protocol_{TAG}.png'
r = subprocess.run([PY, '/content/code/scripts/plot_protocol.py', '--summary', f'{OUT}/summary.json', '--out', fig],
                   capture_output=True, text=True, env=dict(os.environ, PYTHONPATH='/content/code', MPLBACKEND='Agg'))
print(r.stdout[-12000:], r.stderr[-800:])
display(Image(fig)) if os.path.exists(fig) else None


In [ ]:
#@title 8. (optional) Check of the fast approximation on EXACT_ZONES with the same cameras: exact mode (~6 s per draw) and the fast_retest variant
import subprocess, time, os
if EXACT_CHECK and MODE == 'fast':
    env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    for mode, out in (('fast_retest', OUT_RETEST), ('exact', OUT_EXACT)):
        cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
               '--assoc-dir', ASSOC_DIR, '--out', out, '--subset-dir', SUBSET_DIR, '--litept', '/content/LitePT',
               '--backbone-ckpt', CKPT_LOCAL, '--zones', ','.join(exact_zones), '--sweeps', EXACT_SWEEPS, '--draws', str(EXACT_DRAWS),
               '--mode', mode, '--cap', str(CAP), '--cap-full', str(CAP_FULL), '--eval-max-points', str(EVAL_MAX_POINTS), '--seed', str(SEED),
               '--heads'] + [f'{n}={p}' for n, p in heads.items()]
        t0 = time.time()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            if line.startswith('[protocol]') and (' draw ' not in line or mode == 'exact'): print(line, end='')
            elif 'Error' in line or 'Traceback' in line or line.startswith('  File'): print(line, end='')
        p.wait(); print(f"{mode}: exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
    for a, b, names, f in ((OUT, OUT_EXACT, 'fast,exact', 'compare.json'), (OUT_RETEST, OUT_EXACT, 'retest,exact', 'compare_retest.json')):
        r = subprocess.run([PY, '/content/code/scripts/compare_protocol.py', '--a', a, '--b', b, '--names', names, '--out', f'{OUT_EXACT}/{f}'],
                           capture_output=True, text=True, env=env)
        print(r.stdout[-14000:], r.stderr[-800:])
else:
    print("check skipped (EXACT_CHECK off, or MODE is not 'fast')")


In [ ]:
#@title 9. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- `scores.csv`: ต่อ (model, sweep, class) → `clean_iou`, `R` (robustness score, 1 = ไม่ตก), `drop_ref` (IoU ตกที่ reference level: 10 cm / 0.1° / 25 cm drift)
- Gate G3 (run `fast_v1`, 13 พ.ย.): ที่ `trans` 0.10 class `structural` / `conductor` ของทุก head เปลี่ยนไม่เกิน 0.2 IoU (เกณฑ์เรื่อง A คือตก ≥ 15) → **เรื่อง B**
- log ของ cell 6: หนึ่งบรรทัดต่อ (zone, sweep, level, draw) พร้อม mIoU ของโซนนั้น (เฉลี่ยเฉพาะ class ที่โซนนั้นมี) สำหรับทุกโมเดล — `lidar` ต้องคงที่ทุกบรรทัดของโซนเดียวกัน
- ตรวจผล: คอลัมน์ `clean` ของตารางสรุป (cell 7) ต้องเท่ากับ bank estimate ของ heads lab เพราะใช้ subset ชุดเดียวกัน:
  `lidar` 69.9, `imagevote` 65.5, `late_v1` 81.9, `concat_v1` 78.6, `window3_v1` 78.6, `late_nat_b1` 81.8
- cell 8 (`EXACT_CHECK`, เฉพาะเมื่อ `MODE` = fast): รันโหมด exact และ fast_retest บน `EXACT_ZONES` ด้วยกล้องชุดเดียวกับ draw 0..`EXACT_DRAWS`-1 ของโหมด fast
  แล้วพิมพ์ตารางเทียบ → `GridNet/protocol/<TAG>_exact/compare.json` และ `compare_retest.json`
- จบแล้ว **Disconnect and delete runtime**